# Introduction to Deep Learning, Assignment 2, Task 2


# Function definitions for creating the datasets

First we need to create our datasets that are going to be used for training our models.

In order to create image queries of simple arithmetic operations such as '15+13' or '42-10' we need to create images of '+' and '-' signs using ***open-cv*** library. We will use these operand signs together with the MNIST dataset to represent the digits.

In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
import numpy as np
import tensorflow as tf
import random
from sklearn.model_selection import train_test_split

from tensorflow.keras.layers import Dense, RNN, LSTM, Flatten, TimeDistributed, LSTMCell
from tensorflow.keras.layers import RepeatVector, Conv2D, SimpleRNN, GRU, Reshape, ConvLSTM2D, Conv2DTranspose

2025-12-22 14:49:02.062137: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-22 14:49:02.094663: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-22 14:49:11.798675: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
from scipy.ndimage import rotate


# Create plus/minus operand signs
def generate_images(number_of_images=50, sign='-'):
    blank_images = np.zeros([number_of_images, 28, 28])  # Dimensionality matches the size of MNIST images (28x28)
    x = np.random.randint(12, 16, (number_of_images, 2)) # Randomized x coordinates
    y1 = np.random.randint(6, 10, number_of_images)       # Randomized y coordinates
    y2 = np.random.randint(18, 22, number_of_images)     # -||-

    for i in range(number_of_images): # Generate n different images
        cv2.line(blank_images[i], (y1[i], x[i,0]), (y2[i], x[i, 1]), (255,0,0), 2, cv2.LINE_AA)     # Draw lines with randomized coordinates
        if sign == '+':
            cv2.line(blank_images[i], (x[i,0], y1[i]), (x[i, 1], y2[i]), (255,0,0), 2, cv2.LINE_AA) # Draw lines with randomized coordinates

    return blank_images

def show_generated(images, n=5):
    plt.figure(figsize=(2, 2))
    for i in range(n**2):
        plt.subplot(n, n, i+1)
        plt.axis('off')
        plt.imshow(images[i])
    plt.show()

In [3]:
def create_data(highest_integer, num_addends=2, operands=['+', '-']):
    """
    Creates the following data for all pairs of integers up to [1:highest integer][+/-][1:highest_integer]:

    @return:
    X_text: '51+21' -> text query of an arithmetic operation (5)
    X_img : Stack of MNIST images corresponding to the query (5 x 28 x 28) -> sequence of 5 images of size 28x28
    y_text: '72' -> answer of the arithmetic text query
    y_img :  Stack of MNIST images corresponding to the answer (3 x 28 x 28)

    Images for digits are picked randomly from the whole MNIST dataset.
    """

    num_indices = [np.where(MNIST_labels==x) for x in range(10)]
    num_data = [MNIST_data[inds] for inds in num_indices]
    image_mapping = dict(zip(unique_characters[:10], num_data))
    image_mapping['-'] = generate_images()
    image_mapping['+'] = generate_images(sign='+')
    image_mapping['*'] = generate_images(sign='*')
    image_mapping[' '] = np.zeros([1, 28, 28])

    X_text, X_img, y_text, y_img = [], [], [], []

    for i in range(highest_integer + 1):      # First addend
        for j in range(highest_integer + 1):  # Second addend
            for sign in operands: # Create all possible combinations of operands
                query_string = to_padded_chars(str(i) + sign + str(j), max_len=max_query_length, pad_right=True)
                query_image = []
                for n, char in enumerate(query_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    query_image.append(image_set[index].squeeze())

                result = eval(query_string)
                result_string = to_padded_chars(result, max_len=max_answer_length, pad_right=True)
                result_image = []
                for n, char in enumerate(result_string):
                    image_set = image_mapping[char]
                    index = np.random.randint(0, len(image_set), 1)
                    result_image.append(image_set[index].squeeze())

                X_text.append(query_string)
                X_img.append(np.stack(query_image))
                y_text.append(result_string)
                y_img.append(np.stack(result_image))

    return np.stack(X_text), np.stack(X_img)/255., np.stack(y_text), np.stack(y_img)/255.

def to_padded_chars(integer, max_len=3, pad_right=False):
    """
    Returns a string of len()=max_len, containing the integer padded with ' ' on either right or left side
    """
    length = len(str(integer))
    padding = (max_len - length) * ' '
    if pad_right:
        return str(integer) + padding
    else:
        return padding + str(integer)


# Creating our data

The dataset consists of 20000 samples that (additions and subtractions between all 2-digit integers) and they have two kinds of inputs and label modalities:

  **X_text**: strings containing queries of length 5: ['  1+1  ', '11-18', ...]

  **X_image**: a stack of images representing a single query, dimensions: [5, 28, 28]

  **y_text**: strings containing answers of length 3: ['  2', '156']

  **y_image**: a stack of images that represents the answer to a query, dimensions: [3, 28, 28]

In [4]:
# Illustrate the generated query/answer pairs

unique_characters = '0123456789+- '       # All unique characters that are used in the queries (13 in total: digits 0-9, 2 operands [+, -], and a space character ' '.)
highest_integer = 99                      # Highest value of integers contained in the queries

max_int_length = len(str(highest_integer))# Maximum number of characters in an integer
max_query_length = max_int_length * 2 + 1 # Maximum length of the query string (consists of two integers and an operand [e.g. '22+10'])
max_answer_length = 3    # Maximum length of the answer string (the longest resulting query string is ' 1-99'='-98')

# Create the data (might take around a minute)
(MNIST_data, MNIST_labels), _ = tf.keras.datasets.mnist.load_data()
X_text, X_img, y_text, y_img = create_data(highest_integer)
print(X_text.shape, X_img.shape, y_text.shape, y_img.shape)




(20000,) (20000, 5, 28, 28) (20000,) (20000, 3, 28, 28)


# My own helper functions

In the models below teacher forcing is used. For this the vocabulary will need a start and end token. Subsequently the one-hot encoding and decoding functions need to be altered to include these.

In [5]:
# Teacher forcing preparation
vocabulary_tf = list(unique_characters)+['<start>','<end>'] 

indices = {char:i for i, char in enumerate(vocabulary_tf)}
reverse_indices={i:char for i,char in enumerate(vocabulary_tf)}


# One-hot encoding
def encode_labels_tf(labels, vocabulary=vocabulary_tf, indices_map=indices):
  n = len(labels)
  length = len(labels[0])+2 # for start of sequence <start> and end of sequence <end> tokens.

  one_hot = np.zeros([n, length, len(vocabulary)])
  for i, label in enumerate(labels):
    full_label = ['<start>'] + list(label) + ['<end>']
    m = np.zeros([length, len(vocabulary)])
    for j, char in enumerate(full_label):
       m[j, indices_map[char]] = 1
    one_hot[i] = m

  return one_hot

# One-hot decoding
def decode_labels_tf(labels, indices_map=reverse_indices):

    pred_indices = np.argmax(labels, axis=-1) 
    
    decoded_list = []
    for sequence in pred_indices:

        chars = [indices_map[i] for i in sequence if indices_map[i] not in ['<start>', '<end>', '<pad>']]
        decoded_list.append(''.join(chars))
        
    return decoded_list

X_text_onehot = encode_labels_tf(X_text)
y_text_onehot = encode_labels_tf(y_text)

print(X_text_onehot.shape, y_text_onehot.shape)

(20000, 7, 15) (20000, 5, 15)


## Model builder functions

### Pre training model

In [6]:
# Defining the calculator datasets. From X_img to X_text_onehot
size=0.1

X_train_pt, X_test_pt, y_train_pt, y_test_pt = train_test_split(
    X_img, X_text_onehot, random_state=42, test_size=size
)

X_train_pt, X_val_pt, y_train_pt, y_val_pt = train_test_split(
    X_train_pt, y_train_pt, random_state=42, test_size=size/(1-size)
)

max_answer_length_pt = 6

y_train_in_pt = y_train_pt[:, :-1, :]
y_train_target_pt = y_train_pt[:, 1:, :]

y_val_in_pt = y_val_pt[:, :-1, :]
y_val_target_pt = y_val_pt[:, 1:, :]

y_test_in_pt = y_test_pt[:, :-1, :]
y_test_target_pt = y_test_pt[:, 1:, :]

In [7]:
# Your code is: damn code

from tensorflow.keras.layers import BatchNormalization, Activation, MaxPooling2D, LSTM, TimeDistributed, Dropout, Input, Add, LayerNormalization, Attention, Concatenate,GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import L2, L1L2


# First we create a build-encoder function
def build_image2text_encoder(dropout, RLstrength):
    
    # Initialize an encoder
    X_in = Input(shape = (5,28,28,1)) # 5 times a grayscale image


    # Build encoder layers
    ## Block 1
    B1 = TimeDistributed(Conv2D(32, (3,3), padding = "same", kernel_regularizer=L2(RLstrength/8)))(X_in)
    B1 = TimeDistributed(BatchNormalization())(B1)
    B1 = TimeDistributed(Activation('relu'))(B1)
    B1 = TimeDistributed(Dropout(dropout))(B1)
    B1_final = TimeDistributed(MaxPooling2D())(B1)


    ## Initialize the residual connection
    #residual = TimeDistributed(Activation('linear', name = "residual_branch"))(B1_final)
    residual2= TimeDistributed(Conv2D(64, (1,1), kernel_regularizer=L2(RLstrength/8), name = "residual_branch"))(B1_final)
    
    ## Block 2
    B2 = TimeDistributed(Conv2D(64, (3,3), padding = "same", kernel_regularizer=L2(RLstrength/8)))(B1_final)
    B2 = TimeDistributed(BatchNormalization())(B2)
    B2 = TimeDistributed(Activation('relu'))(B2)
    B2_final = TimeDistributed(Dropout(dropout))(B2)

    ## Connect residual connection to output of two Conv2D blocks
    combined = Add()([B2_final, residual2])
    combined = TimeDistributed(Activation('relu'))(combined)

    ## Final pooling before the ConvLSTM2D layer
    final_pooling = TimeDistributed(MaxPooling2D(name='final_pooling'))(combined)


    ## Recurrent convolutional layers
    output, hidden, cell = ConvLSTM2D(
            filters=128, 
            kernel_size=(3,3), 
            padding='same',
            return_sequences=True, 
            use_bias=True, 
            return_state=True, 
            name='ConvLSTM', 
            dropout=dropout, #dropout
            #recurrent_dropout=dropout, #dropout
            kernel_regularizer = L2(RLstrength),
            recurrent_regularizer = L2(RLstrength))(final_pooling) #L2(RLstrength)

    encoder = tf.keras.Model(inputs=X_in, outputs=[output, hidden, cell], name = "encoder_model")
    return encoder

In [8]:
def build_image2text_pretraining(dropout = 0.5, max_size=512,RLstrength=1.0e-4):

    vocab_size = 15

    X_in = Input(shape = (5,28,28,1), name = 'sequence')
    Y_in = Input(shape=(6, vocab_size))

    encoder = build_image2text_encoder(dropout,RLstrength)
    _, hidden, cell = encoder(X_in)
    
    h_flattened = GlobalAveragePooling2D(name='h_flattened')(hidden)#Flatten()(hidden)
    h_initial = Dense(max_size, kernel_regularizer=L2(RLstrength), name='h0')(h_flattened)

    c_flattened = GlobalAveragePooling2D(name='c_flattened')(cell)#Flatten(name='c_flattened')(cell)
    c_initial = Dense(max_size, kernel_regularizer=L2(RLstrength),name='c0')(c_flattened)

    ini_state = [h_initial, c_initial]
    
    output, hidden, cell = LSTM(
        max_size, 
        return_sequences = True,
        return_state=True, 
        dropout = dropout, 
        #recurrent_dropout = dropout, 
        name='lstm_pre_training',
        kernel_regularizer = L2(RLstrength/4),
        recurrent_regularizer = L2(RLstrength/4)
        )(Y_in, initial_state = ini_state)
        
    dense = TimeDistributed(Dense(vocab_size, activation='softmax', name = 'decoder_dense_pre_training'))
    y_out = dense(output)

    full = tf.keras.Model(inputs=[X_in, Y_in], outputs = y_out, name = 'pretraining_model')
#    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
#    full.compile(
#        loss=loss, optimizer=Adam(learning_rate=learning_rate), metrics=['categorical_accuracy']
#    )

    full.summary(expand_nested=True)
    return full



### Calculator model

In [9]:
# Defining the calculator datasets. From X_text_onehot to y_text_onehot
size=0.1

X_train_calc, X_test_calc, y_train_calc, y_test_calc = train_test_split(
    X_text_onehot[:, :-1, :], y_text_onehot, random_state=42, test_size=size
)

X_train_calc, X_val_calc, y_train_calc, y_val_calc = train_test_split(
    X_train_calc, y_train_calc, random_state=42, test_size=size/(1-size)
)

max_answer_length_tf = 4

y_train_in_calc = y_train_calc[:, :-1, :]
y_train_target_calc = y_train_calc[:, 1:, :]

y_val_in_calc = y_val_calc[:, :-1, :]
y_val_target_calc = y_val_calc[:, 1:, :]

y_test_in_calc = y_test_calc[:, :-1, :]
y_test_target_calc = y_test_calc[:, 1:, :]

In [10]:
# Here we build the full model
def build_text2text_calc(dropout = 0.5, max_size=512, learning_rate = 2.5e-4, max_answer_length_tf=4,RLstrength=1.0e-4):

    vocab_size = len(vocabulary_tf)

    # Define input layer of full model
    X_in = Input(shape = (6, vocab_size), name = 'expression_input')
    Y_in = Input(shape=(max_answer_length_tf, len(vocabulary_tf)), name = "answer")


    # calculator encoder
    encoder_lstm = LSTM(max_size, return_state=True, return_sequences=True, name = 'calculator_encoder')
    key, hidden, cell = encoder_lstm(X_in)
    ini_state = [hidden, cell]    


    decoder_lstm = LSTM(
        max_size, 
        return_sequences = True, 
        return_state=True, 
        dropout = dropout, 
        recurrent_dropout = dropout, 
        name='decoder_lstm',
        kernel_regularizer = L2(RLstrength/4),
        recurrent_regularizer = L2(RLstrength/4)
        ) #
    

    query, _, _ = decoder_lstm(Y_in, initial_state = ini_state)

    attention_block = Attention(name='attention_block')([query, key])

    combined = Concatenate(axis=-1, name = 'concat_q_A')([query, attention_block])
    combined = Dense(max_size, name = 'combined_dense', kernel_regularizer = L1L2(l1 = 5.0e-5, l2=RLstrength/2))(combined)

    residual = TimeDistributed(Dense(
        max_size, 
        use_bias=False, 
        kernel_regularizer = L2(RLstrength/2)
        ), 
        name = 'decoder_res_dense'
        )(Y_in)

    final = Add(name= 'decoder_with_residual')([combined, residual])
    final = TimeDistributed(Activation('relu'), name = "decoder_activation")(final)
    final = LayerNormalization(axis=-1, name='decoder_layer_norm')(final)
    
    y_out = TimeDistributed(Dense(vocab_size, activation='softmax'), name='decoder_dense')(final)


    # Full model step
    full = tf.keras.Model(inputs=[X_in, Y_in], outputs = y_out, name = 'calculator')
#    full.compile(
#        loss='categorical_crossentropy', optimizer=AdamW(learning_rate=learning_rate), metrics=['categorical_accuracy']
#    )

    full.summary(expand_nested=True)
    return full

### Concatenate visual encoder and calculator

In [11]:
# Defining the full datasets. From X_img to y_text_onehot
size=0.1

X_train, X_test, y_train, y_test = train_test_split(
    X_img, y_text_onehot, random_state=42, test_size=size
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, random_state=42, test_size=size/(1-size)
)

y_train_in = y_train[:, :-1, :]
y_train_target = y_train[:, 1:, :]

y_val_in = y_val[:, :-1, :]
y_val_target = y_val[:, 1:, :]

y_test_in = y_test[:, :-1, :]
y_test_target = y_test[:, 1:, :]

In [29]:
import tensorflow as tf
from tensorflow.keras.layers import Input, TimeDistributed, Dense, GlobalAveragePooling2D, Reshape
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import AdamW

def build_grafted_model_pooled(vision_model, calc_model, learning_rate=1e-5):
    """
    Grafts Vision to Calculator using POOLING (Sequence Length 5).
    This matches the Calculator's expected attention span much better.
    """
    
    # --- 1. Fresh Inputs ---
    new_image_input = Input(shape=(5, 28, 28, 1), name='graft_img_input')
    new_answer_input = Input(shape=(4, 15), name='graft_ans_input')

    # --- 2. Vision Encoding ---
    # Re-call the encoder
    encoder_instance = vision_model.get_layer('encoder_model')
    raw_seq, raw_h, raw_c = encoder_instance(new_image_input)

    # --- 3. Manifold Alignment (The Bridge) ---
    
    # A. States (Initialize the Brain)
    # Pool spatial dims: (Batch, 7, 7, 128) -> (Batch, 128)
    h_pooled = GlobalAveragePooling2D(name='graft_h_pool')(raw_h)
    c_pooled = GlobalAveragePooling2D(name='graft_c_pool')(raw_c)
    
    # Project to 256
    h_init = Dense(256, activation='tanh', name='graft_h_proj')(h_pooled)
    c_init = Dense(256, activation='tanh', name='graft_c_proj')(c_pooled)

    # B. Sequence (The Eyes) -> CRITICAL CHANGE HERE
    # Instead of flattening to 245, we Pool to 5.
    # Input: (Batch, 5, 7, 7, 128) -> Output: (Batch, 5, 128)
    seq_pooled = TimeDistributed(
        GlobalAveragePooling2D(), 
        name='graft_seq_pool'
    )(raw_seq)
    
    # Project to 256
    seq_projected = TimeDistributed(
        Dense(256, activation='tanh'), 
        name='graft_seq_proj'
    )(seq_pooled)

    # --- 4. Calculator Decoder ---
    decoder_lstm = calc_model.get_layer('decoder_lstm')
    
    # Initialize with Vision States
    decoder_output, _, _ = decoder_lstm(
        new_answer_input, 
        initial_state=[h_init, c_init]
    )

    # --- 5. Attention ---
    attention_block = calc_model.get_layer('attention_block')
    
    # Now Attention compares (Batch, 4, 256) vs (Batch, 5, 256)
    # This is an easy task for the pre-trained weights!
    context_vector = attention_block([decoder_output, seq_projected])

    # --- 6. Reconstruction ---
    concat_layer = calc_model.get_layer('concat_q_A')
    combined = concat_layer([decoder_output, context_vector])
    
    combined_dense = calc_model.get_layer('combined_dense')
    combined = combined_dense(combined)
    
    res_dense = calc_model.get_layer('decoder_res_dense')
    residual_path = res_dense(new_answer_input)
    
    add_layer = calc_model.get_layer('decoder_with_residual')
    final_combined = add_layer([combined, residual_path])
    
    act_layer = calc_model.get_layer('decoder_activation')
    norm_layer = calc_model.get_layer('decoder_layer_norm')
    output_dense = calc_model.get_layer('decoder_dense')
    
    final_output = output_dense(norm_layer(act_layer(final_combined)))

    # --- 7. Compile ---
    model = Model(
        inputs=[new_image_input, new_answer_input], 
        outputs=final_output, 
        name='grafted_calculator_pooled'
    )
    
    model.compile(
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        optimizer=AdamW(learning_rate=learning_rate, weight_decay=1e-4), 
        metrics=['categorical_accuracy']
    )
    
    return model

## Training the models

In [30]:
# Training the visual encoder. We train this beforehand.
## Building model
dropout=0.5
RLstrength=0
max_size=256
image2text_pretraining = build_image2text_pretraining(dropout=dropout, RLstrength=RLstrength, max_size=max_size)


## Compile
learning_rate=4.0e-4 # Initial LR
weight_decay=3.0e-4  # Decoupled Weight Decay

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=learning_rate,
    weight_decay=weight_decay,
)

loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

image2text_pretraining.compile(
    loss=loss, 
    optimizer=optimizer, 
    metrics=['categorical_accuracy'])


## Training
stopper_patience_warmup = 10

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience_warmup,
    restore_best_weights=True
)

### Warm-start
history = image2text_pretraining.fit(x=[X_train_pt, y_train_in_pt], y=y_train_target_pt, 
               epochs = 50,
               batch_size = 32,
               validation_data = ([X_val_pt, y_val_in_pt], y_val_target_pt),
               callbacks = [early_stopper])

### Recompile so AdamW moments are reset
image2text_pretraining.compile(
    loss=loss, 
    optimizer=optimizer, 
    metrics=['categorical_accuracy'])


scheduler_patience = 5
stopper_patience = 20

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=scheduler_patience,
    min_lr=1e-6,
    verbose=1
)

### Fine-tune training
history = image2text_pretraining.fit(x=[X_train_pt, y_train_in_pt], y=y_train_target_pt, 
               epochs = 60,
               batch_size = 32,
               validation_data = ([X_val_pt, y_val_in_pt], y_val_target_pt),
               callbacks=[lr_scheduler, early_stopper],
               verbose=0)


Model: "pretraining_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence            │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_model       │ [(None, 5, 7, 7,  │    906,560 │ sequence[0][0]    │
│ (Functional)        │ 128), (None, 7,   │            │                   │
│                     │ 7, 128), (None,   │            │                   │
│                     │ 7, 7, 128)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ input_layer_5  │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │        320 │ -                 │
│ time_distributed_28 │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │        128 │ -                 │
│ time_distributed_29 │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_30 │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 28, 28, │          0 │ -                 │
│ time_distributed_31 │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_32 │ 32)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │     18,496 │ -                 │
│ time_distributed_34 │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │        256 │ -                 │
│ time_distributed_35 │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_36 │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │          0 │ -                 │
│ time_distributed_37 │ 64)               │            │                   │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 5, 14, 14, │      2,112 │ -                 │
│ time_distributed_33 │ 64)               │            │                 

 Total params: 1,254,991 (4.79 MB)

 Trainable params: 1,254,799 (4.79 MB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/50


E0000 00:00:1766416041.950334 1146156 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Sigmoid' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_5', 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Sigmoid_2' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_6', 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/

500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 39ms/step - categorical_accuracy: 0.4249 - loss: 1.8191 - val_categorical_accuracy: 0.3986 - val_loss: 1.9241
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 39ms/step - categorical_accuracy: 0.6106 - loss: 1.4159 - val_categorical_accuracy: 0.5809 - val_loss: 1.5537
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 16s 32ms/step - categorical_accuracy: 0.7888 - loss: 1.0949 - val_categorical_accuracy: 0.6670 - val_loss: 1.3629
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 17s 33ms/step - categorical_accuracy: 0.8889 - loss: 0.8907 - val_categorical_accuracy: 0.6363 - val_loss: 1.5281
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 17s 34ms/step - categorical_accuracy: 0.9338 - loss: 0.7821 - val_categorical_accuracy: 0.7146 - val_loss: 1.2767
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - categorical_accuracy: 0.9530 - loss: 0.7285 - val_categorical_accuracy: 0.7784 - val_loss: 1.1210
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 19s 38ms/step - categorical_accuracy: 0.9639 - 

E0000 00:00:1766416996.060090 1146156 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_4' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_7', 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Sigmoid' -> 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_5', 'StatefulPartitionedCall/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/body/_87/pretraining_model_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/conv


Epoch 12: ReduceLROnPlateau reducing learning rate to 0.00019999999494757503.

Epoch 17: ReduceLROnPlateau reducing learning rate to 9.999999747378752e-05.

Epoch 22: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.

Epoch 27: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-05.


In [31]:
# Training the calculator

## Building model
dropout=0.5
RLstrength=0
max_size=256
text2text_calculator = build_text2text_calc(dropout=dropout, RLstrength=RLstrength, max_size=max_size)

## Compile
learning_rate=5.0e-4 
weight_decay=1.0e-4  

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=learning_rate,
    weight_decay=weight_decay,
)

loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

text2text_calculator.compile(
    loss=loss, 
    optimizer=optimizer, 
    metrics=['categorical_accuracy'])

## Training
stopper_patience_warmup = 5

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience_warmup,
    restore_best_weights=True
)

### Warm-start
history_warmup = text2text_calculator.fit(x=[X_train_calc, y_train_in_calc], y=y_train_target_calc, 
               epochs = 50,
               batch_size = 32,
               validation_data = ([X_val_calc, y_val_in_calc], y_val_target_calc),
               callbacks = [early_stopper])

### Recompile so AdamW momenta are reset
text2text_calculator.compile(
    loss=loss, 
    optimizer=optimizer, 
    metrics=['categorical_accuracy'])

scheduler_patience = 3
stopper_patience = 12

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor = 'val_loss',
    patience=stopper_patience,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=scheduler_patience,
    min_lr=1e-6,
    verbose=1
)

### Fine-tune training
history_calc = text2text_calculator.fit(x=[X_train_calc, y_train_in_calc], y=y_train_target_calc, 
               epochs = 60,
               batch_size = 32,
               validation_data = ([X_val_calc, y_val_in_calc], y_val_target_calc),
               callbacks=[lr_scheduler, early_stopper],
               verbose=0)

Model: "calculator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ expression_input    │ (None, 6, 15)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ answer (InputLayer) │ (None, 4, 15)     │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ calculator_encoder  │ [(None, 6, 256),  │    278,528 │ expression_input… │
│ (LSTM)              │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 4, 256),  │    278,528 │ answer[0][0],     │
│                     │ (None, 256),      │            │ calculator_encod… │
│                     │ (None, 256)]      │            │ calculator_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_block     │ (None, 4, 256)    │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ calculator_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_q_A          │ (None, 4, 512)    │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ attention_block[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined_dense      │ (None, 4, 256)    │    131,328 │ concat_q_A[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_res_dense   │ (None, 4, 256)    │      3,840 │ answer[0][0]      │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_with_resid… │ (None, 4, 256)    │          0 │ combined_dense[0… │
│ (Add)               │                   │            │ decoder_res_dens… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_activation  │ (None, 4, 256)    │          0 │ decoder_with_res… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_layer_norm  │ (None, 4, 256)    │        512 │ decoder_activati… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, 4, 15)     │      3,855 │ decoder_layer_no… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 696,591 (2.66 MB)

 Trainable params: 696,591 (2.66 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - categorical_accuracy: 0.5380 - loss: 1.9434 - val_categorical_accuracy: 0.5670 - val_loss: 1.7645
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - categorical_accuracy: 0.5841 - loss: 1.6851 - val_categorical_accuracy: 0.5980 - val_loss: 1.6023
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - categorical_accuracy: 0.6291 - loss: 1.5207 - val_categorical_accuracy: 0.6498 - val_loss: 1.4467
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - categorical_accuracy: 0.6597 - loss: 1.4162 - val_categorical_accuracy: 0.6839 - val_loss: 1.3569
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - categorical_accuracy: 0.6865 - loss: 1.3389 - val_categorical_accuracy: 0.7158 - val_loss: 1.2887
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - categorical_accuracy: 0.7050 - loss: 1.2848 - val_categorical_accuracy: 0.7273 - val_loss: 1.2470
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - categorical_accuracy: 0.72

In [26]:
# Building full model
image2text = build_grafted_model(image2text_pretraining, text2text_calculator)
#full_model.summary()

# Phase 1: Static visual encoder weights
image2text_pretraining.trainable = False

learning_rate_static = 5e-4
weight_decay_static = 1e-4

optimizer_static = tf.keras.optimizers.AdamW(
    learning_rate=learning_rate_static,
    weight_decay=weight_decay_static
)

loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

image2text.compile(
    optimizer=optimizer_static, 
    loss=loss_fn,
    metrics=['categorical_accuracy']
)

image2text.summary(expand_nested=False)

history_full_static = image2text.fit(
    x=[X_train, y_train_in], 
    y=y_train_target, 
    epochs=7,
    batch_size=32,
    validation_data=([X_val, y_val_in], y_val_target)
)

# Phase 2: Dynamic visual encoder weights
image2text_pretraining.trainable = True

learning_rate_dynamic = 1.0e-5
weight_decay_dynamic = 5e-4 

optimizer_dynamic = tf.keras.optimizers.AdamW(
    learning_rate=learning_rate_dynamic,
    weight_decay=weight_decay_dynamic
)

image2text.compile(
    optimizer=optimizer_dynamic, 
    loss=loss_fn,
    metrics=['categorical_accuracy']
)

stopper_patience = 25
scheduler_patience = 5

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=stopper_patience,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=scheduler_patience,
    min_lr=1e-6,
    verbose=1
)

history_full_dynamic = image2text.fit(
    x=[X_train, y_train_in], 
    y=y_train_target, 
    epochs=60,
    batch_size=32,
    validation_data=([X_val, y_val_in], y_val_target),
    callbacks=[lr_scheduler, early_stopper]
)

Model: "grafted_calculator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ graft_img_input     │ (None, 5, 28, 28, │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_model       │ [(None, 5, 7, 7,  │    906,560 │ graft_img_input[… │
│ (Functional)        │ 128), (None, 7,   │            │                   │
│                     │ 7, 128), (None,   │            │                   │
│                     │ 7, 7, 128)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ graft_h_pool        │ (None, 128)       │          0 │ encoder_model[2]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ graft_c_pool        │ (None, 128)       │          0 │ encoder_model[2]… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ graft_ans_input     │ (None, 4, 15)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ graft_h_proj        │ (None, 256)       │     33,024 │ graft_h_pool[0][… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ graft_c_proj        │ (None, 256)       │     33,024 │ graft_c_pool[0][… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ graft_seq_flatten   │ (None, 245, 128)  │          0 │ encoder_model[2]… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 4, 256),  │    278,528 │ graft_ans_input[… │
│                     │ (None, 256),      │            │ graft_h_proj[0][… │
│                     │ (None, 256)]      │            │ graft_c_proj[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ graft_seq_proj      │ (None, 245, 256)  │     33,024 │ graft_seq_flatte… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_block     │ (None, 4, 256)    │          0 │ decoder_lstm[2][… │
│ (Attention)         │                   │            │ graft_seq_proj[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_q_A          │ (None, 4, 512)    │          0 │ decoder_lstm[2][… │
│ (Concatenate)       │                   │            │ attention_block[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined_dense      │ (None, 4, 256)    │    131,328 │ concat_q_A[2][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_res_dense   │ (None, 4, 256)    │      3,840 │ graft_ans_input[… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_with_resid… │ (None, 4, 256)    │          0 │ combined_dense[2… │
│ (Add)               │                   │            │ decoder_res_dens… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_activation  │ (None, 4, 256)    │          0 │ decoder_with_res

 Total params: 1,423,695 (5.43 MB)

 Trainable params: 517,135 (1.97 MB)

 Non-trainable params: 906,560 (3.46 MB)

Epoch 1/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 21s 32ms/step - categorical_accuracy: 0.5635 - loss: 1.5961 - val_categorical_accuracy: 0.5710 - val_loss: 1.5463
Epoch 2/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 15s 29ms/step - categorical_accuracy: 0.6044 - loss: 1.4560 - val_categorical_accuracy: 0.5738 - val_loss: 1.5461
Epoch 3/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 14s 29ms/step - categorical_accuracy: 0.6262 - loss: 1.4010 - val_categorical_accuracy: 0.5896 - val_loss: 1.5006
Epoch 4/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 15s 29ms/step - categorical_accuracy: 0.6399 - loss: 1.3670 - val_categorical_accuracy: 0.6026 - val_loss: 1.4794
Epoch 5/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 14s 29ms/step - categorical_accuracy: 0.6500 - loss: 1.3423 - val_categorical_accuracy: 0.6026 - val_loss: 1.4971
Epoch 6/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 14s 29ms/step - categorical_accuracy: 0.6587 - loss: 1.3217 - val_categorical_accuracy: 0.6162 - val_loss: 1.4574
Epoch 7/7
500/500 ━━━━━━━━━━━━━━━━━━━━ 15s 29ms/step - categorical_accuracy: 0.666

E0000 00:00:1766415374.332749 1146156 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/grafted_calculator_1/encoder_model_1/ConvLSTM_1/while/body/_108/grafted_calculator_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/convolution_6' -> 'StatefulPartitionedCall/grafted_calculator_1/encoder_model_1/ConvLSTM_1/while/body/_108/grafted_calculator_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_6', 'StatefulPartitionedCall/grafted_calculator_1/encoder_model_1/ConvLSTM_1/while/body/_108/grafted_calculator_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_4' -> 'StatefulPartitionedCall/grafted_calculator_1/encoder_model_1/ConvLSTM_1/while/body/_108/grafted_calculator_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_7', 'StatefulPartitionedCall/grafted_calculator_1/encoder_model_1/ConvLSTM_1/while/body/_108/grafted_calculator_1/encoder_model_1/ConvLSTM_1/while/

500/500 ━━━━━━━━━━━━━━━━━━━━ 38s 62ms/step - categorical_accuracy: 0.6829 - loss: 1.2754 - val_categorical_accuracy: 0.6034 - val_loss: 1.4910 - learning_rate: 1.0000e-05
Epoch 2/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 30s 60ms/step - categorical_accuracy: 0.6871 - loss: 1.2667 - val_categorical_accuracy: 0.6091 - val_loss: 1.4800 - learning_rate: 1.0000e-05
Epoch 3/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 30s 60ms/step - categorical_accuracy: 0.6888 - loss: 1.2635 - val_categorical_accuracy: 0.6107 - val_loss: 1.4776 - learning_rate: 1.0000e-05
Epoch 4/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 30s 60ms/step - categorical_accuracy: 0.6904 - loss: 1.2587 - val_categorical_accuracy: 0.6098 - val_loss: 1.4762 - learning_rate: 1.0000e-05
Epoch 5/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 30s 60ms/step - categorical_accuracy: 0.6907 - loss: 1.2579 - val_categorical_accuracy: 0.6096 - val_loss: 1.4779 - learning_rate: 1.0000e-05
Epoch 6/60
500/500 ━━━━━━━━━━━━━━━━━━━━ 30s 60ms/step - categorical_accuracy: 0.6921 - loss: 1.2556 -

KeyboardInterrupt: 

In [32]:
# 1. Build the model
grafted_model = build_grafted_model_pooled(image2text_pretraining, text2text_calculator)

# 2. Freeze the giants
image2text_pretraining.trainable = False
text2text_calculator.trainable = False

# Note: The new Dense layers created INSIDE the function are NOT part of those models, 
# so they remain trainable by default. This is exactly what we want.

# 3. Recompile to apply freezing
grafted_model.compile(
    loss='categorical_crossentropy', 
    optimizer=AdamW(learning_rate=1e-3), # Higher LR to learn the projection quickly
    metrics=['categorical_accuracy']
)

# 4. Train for a short burst (e.g., 5 epochs)
print("--- Phase A: Alignment ---")
grafted_model.fit([X_train, y_train_in], y_train_target, epochs=7, batch_size=64)

# 1. Unfreeze
image2text_pretraining.trainable = True
text2text_calculator.trainable = True

# 2. Recompile with LOW Learning Rate
grafted_model.compile(
    loss='categorical_crossentropy', 
    optimizer=AdamW(learning_rate=1e-5), # Very low LR to preserve knowledge
    metrics=['categorical_accuracy']
)

# 3. Train for real
print("--- Phase B: Fine-Tuning ---")
history = grafted_model.fit(
    [X_train, y_train_in], 
    y_train_target, 
    epochs=50, 
    callbacks=[early_stopper, lr_scheduler]
)

--- Phase A: Alignment ---
Epoch 1/7
250/250 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - categorical_accuracy: 0.5310 - loss: 1.6117
Epoch 2/7
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - categorical_accuracy: 0.5527 - loss: 1.4396
Epoch 3/7
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - categorical_accuracy: 0.5616 - loss: 1.3869
Epoch 4/7
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - categorical_accuracy: 0.5715 - loss: 1.3525
Epoch 5/7
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - categorical_accuracy: 0.5790 - loss: 1.3289
Epoch 6/7
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - categorical_accuracy: 0.5860 - loss: 1.3026
Epoch 7/7
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - categorical_accuracy: 0.5899 - loss: 1.2871
--- Phase B: Fine-Tuning ---
Epoch 1/50


E0000 00:00:1766418377.963891 1146156 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/grafted_calculator_pooled_1/encoder_model_1/ConvLSTM_1/while/body/_108/grafted_calculator_pooled_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_4' -> 'StatefulPartitionedCall/grafted_calculator_pooled_1/encoder_model_1/ConvLSTM_1/while/body/_108/grafted_calculator_pooled_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_7', 'StatefulPartitionedCall/gradient_tape/grafted_calculator_pooled_1/encoder_model_1/ConvLSTM_1/while/grafted_calculator_pooled_1/encoder_model_1/ConvLSTM_1/while_grad/body/_924/gradient_tape/grafted_calculator_pooled_1/encoder_model_1/ConvLSTM_1/while/gradients/grafted_calculator_pooled_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Tanh_1_grad/TanhGrad' -> 'StatefulPartitionedCall/gradient_tape/grafted_calculator_pooled_1/encoder_model_1/ConvLSTM_1/whi

500/500 ━━━━━━━━━━━━━━━━━━━━ 25s 42ms/step - categorical_accuracy: 0.6015 - loss: 1.2102 - learning_rate: 1.0000e-05
Epoch 2/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 21s 42ms/step - categorical_accuracy: 0.6048 - loss: 1.1623 - learning_rate: 1.0000e-05
Epoch 3/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 21s 42ms/step - categorical_accuracy: 0.6092 - loss: 1.1370 - learning_rate: 1.0000e-05
Epoch 4/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 21s 42ms/step - categorical_accuracy: 0.6127 - loss: 1.1198 - learning_rate: 1.0000e-05
Epoch 5/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 21s 42ms/step - categorical_accuracy: 0.6170 - loss: 1.1062 - learning_rate: 1.0000e-05
Epoch 6/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 21s 42ms/step - categorical_accuracy: 0.6206 - loss: 1.0939 - learning_rate: 1.0000e-05
Epoch 7/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 21s 42ms/step - categorical_accuracy: 0.6213 - loss: 1.0845 - learning_rate: 1.0000e-05
Epoch 8/50
500/500 ━━━━━━━━━━━━━━━━━━━━ 21s 42ms/step - categorical_accuracy: 0.6246 - loss: 1.0759 - learning_rate

In [36]:
grafted_model.compile(
    loss='categorical_crossentropy', 
    optimizer=AdamW(learning_rate=4e-5), # Very low LR to preserve knowledge
    metrics=['categorical_accuracy']
)


stopper_patience = 25
scheduler_patience = 5

early_stopper = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=stopper_patience,
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=scheduler_patience,
    min_lr=1e-6,
    verbose=1
)

# 3. Train for real
print("--- Phase B: Fine-Tuning ---")
history = grafted_model.fit(
    x=[X_train, y_train_in], 
    y=y_train_target, 
    epochs=100,
    batch_size=32,
    validation_data=([X_val, y_val_in], y_val_target),
    callbacks=[lr_scheduler, early_stopper]
)

--- Phase B: Fine-Tuning ---
Epoch 1/100


E0000 00:00:1766423516.287501 1146156 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/grafted_calculator_pooled_1/encoder_model_1/ConvLSTM_1/while/body/_108/grafted_calculator_pooled_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/mul_4' -> 'StatefulPartitionedCall/grafted_calculator_pooled_1/encoder_model_1/ConvLSTM_1/while/body/_108/grafted_calculator_pooled_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/add_7', 'StatefulPartitionedCall/gradient_tape/grafted_calculator_pooled_1/encoder_model_1/ConvLSTM_1/while/grafted_calculator_pooled_1/encoder_model_1/ConvLSTM_1/while_grad/body/_924/gradient_tape/grafted_calculator_pooled_1/encoder_model_1/ConvLSTM_1/while/gradients/grafted_calculator_pooled_1/encoder_model_1/ConvLSTM_1/while/conv_lstm_cell_1/Tanh_1_grad/TanhGrad' -> 'StatefulPartitionedCall/gradient_tape/grafted_calculator_pooled_1/encoder_model_1/ConvLSTM_1/whi

500/500 ━━━━━━━━━━━━━━━━━━━━ 26s 45ms/step - categorical_accuracy: 0.8841 - loss: 0.3400 - val_categorical_accuracy: 0.7154 - val_loss: 0.9722 - learning_rate: 4.0000e-05
Epoch 2/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 44ms/step - categorical_accuracy: 0.8842 - loss: 0.3357 - val_categorical_accuracy: 0.7183 - val_loss: 0.9652 - learning_rate: 4.0000e-05
Epoch 3/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 43ms/step - categorical_accuracy: 0.8862 - loss: 0.3321 - val_categorical_accuracy: 0.7230 - val_loss: 0.9330 - learning_rate: 4.0000e-05
Epoch 4/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 18s 36ms/step - categorical_accuracy: 0.8882 - loss: 0.3278 - val_categorical_accuracy: 0.7125 - val_loss: 1.0024 - learning_rate: 4.0000e-05
Epoch 5/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 18s 37ms/step - categorical_accuracy: 0.8879 - loss: 0.3276 - val_categorical_accuracy: 0.7118 - val_loss: 0.9885 - learning_rate: 4.0000e-05
Epoch 6/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 22s 43ms/step - categorical_accuracy: 0.8920 - loss: 0.3

## Inference

In [41]:
import numpy as np

def evaluate_grafted_model(model, test_images, test_targets, index_to_char=reverse_indices, 
                           batch_size=64, max_len=4, 
                           start_token='<start>', end_token='<end>', pad_token='<pad>'):
    """
    Performs batch inference on the end-to-end grafted model.
    
    Args:
        model: The compiled 'grafted_calculator' model.
        test_images: (N, 5, 28, 28, 1) array of test images.
        test_targets: (N, max_len, vocab_size) one-hot array of ground truth.
        index_to_char: Dict mapping indices to characters.
        batch_size: Number of samples to process at once (e.g., 64).
    
    Returns:
        accuracy: Exact Match (EM) accuracy.
        predictions: List of decoded strings.
    """
    num_samples = test_images.shape[0]
    vocab_size = len(index_to_char)
    char_to_index = {v: k for k, v in index_to_char.items()}
    start_idx = char_to_index[start_token]
    
    # Store results
    all_pred_strings = []
    correct_count = 0
    
    # --- OUTER LOOP: Iterate over the dataset in batches ---
    print(f"Starting inference on {num_samples} samples...")
    
    for i in range(0, num_samples, batch_size):
        # 1. Prepare the Batch
        batch_end = min(i + batch_size, num_samples)
        batch_imgs = test_images[i:batch_end]
        current_batch_size = batch_end - i
        
        # 2. Initialize Decoder Input (The "Answer" placeholder)
        # Shape: (Batch, 4, 15) - Filled with zeros/padding initially
        batch_answers = np.zeros((current_batch_size, max_len, vocab_size))
        
        # Set the first token to <start> for the entire batch
        batch_answers[:, 0, start_idx] = 1.0
        
        # Track which sequences in the batch are finished
        batch_finished = np.zeros(current_batch_size, dtype=bool)
        batch_indices = np.zeros((current_batch_size, max_len), dtype=int)
        batch_indices[:, 0] = start_idx
        
        # --- INNER LOOP: Autoregressive Decoding (Token by Token) ---
        for t in range(max_len - 1):
            # Predict step t
            # Note: We pass the WHOLE batch of images and current partial answers
            # The model re-processes the images every step (see note below)
            preds = model.predict([batch_imgs, batch_answers], batch_size=current_batch_size, verbose=0)
            
            # Greedy Decode: Get the token with highest probability for step t
            next_tokens = np.argmax(preds[:, t, :], axis=-1)
            
            # Update the inputs for step t+1
            for b in range(current_batch_size):
                if not batch_finished[b]:
                    token = next_tokens[b]
                    batch_indices[b, t + 1] = token
                    
                    # Update One-Hot Vector for next step's input
                    batch_answers[b, t + 1, token] = 1.0
                    
                    if index_to_char.get(token) == end_token:
                        batch_finished[b] = True
            
            if np.all(batch_finished):
                break
        
        # --- SCORING: Compare Batch Predictions to Targets ---
        batch_targets = test_targets[i:batch_end]
        true_indices = np.argmax(batch_targets, axis=-1)
        
        for b in range(current_batch_size):
            # Reconstruct String (Prediction)
            p_chars = [index_to_char[idx] for idx in batch_indices[b] 
                       if index_to_char[idx] not in [start_token, end_token, pad_token]]
            p_str = "".join(p_chars)
            all_pred_strings.append(p_str)
            
            # Reconstruct String (Ground Truth)
            t_chars = [index_to_char[idx] for idx in true_indices[b] 
                       if index_to_char[idx] not in [start_token, end_token, pad_token]]
            t_str = "".join(t_chars)
            
            if p_str == t_str:
                correct_count += 1
        
        # Optional: Print progress
        if (i // batch_size) % 10 == 0:
            print(f"Processed {batch_end}/{num_samples} samples...")

    accuracy = correct_count / num_samples
    print(f"\nFinal Exact Match Accuracy: {accuracy:.2%}")
    
    return accuracy, all_pred_strings

In [42]:
# Assuming you have X_test and y_test_onehot ready
acc, preds = evaluate_grafted_model(
    model=full_model,
    test_images=X_test,
    test_targets=y_test,
#    index_to_char=index_to_char,
    batch_size=64, # Adjust based on your GPU VRAM
    max_len=4
)

# Inspect a few results
for k in range(5):
    print(f"Sample {k}: Predicted '{preds[k]}'")

Starting inference on 2000 samples...
Processed 64/2000 samples...
Processed 704/2000 samples...
Processed 1344/2000 samples...
Processed 1984/2000 samples...

Final Exact Match Accuracy: 0.05%
Sample 0: Predicted '-24'
Sample 1: Predicted '00'
Sample 2: Predicted '24'
Sample 3: Predicted '-4'
Sample 4: Predicted '42'


In [ ]:
# Defining the inference procedure

def predict_math_expression(image_sequence, model, index_to_char, 
                            start_token='<start>', end_token='<end>', 
                            max_len=4, vocab_size=15):

    # Prepare Inputs
    img_input = np.expand_dims(image_sequence, axis=0)
    
    # Initialize decoder input with zeros
    decoder_input = np.zeros((1, max_len, vocab_size))
    
    # Find the index for start token
    char_to_index = {v: k for k, v in index_to_char.items()}
    start_idx = char_to_index[start_token]
    end_idx = char_to_index[end_token]
    
    # Seed the first position with <start>
    decoder_input[0, 0, start_idx] = 1.0
    
    predicted_indices = [start_idx]
    
    # Recursive Loop
    for i in range(max_len - 1):
        # Predict the next token
        preds = model.predict([img_input, decoder_input], verbose=0)
        
        # Get the character at the current step (i+1)
        next_idx = np.argmax(preds[0, i, :])
        predicted_indices.append(next_idx)
        
        # Stop if we hit the end token
        if next_idx == end_idx:
            break
            
        # Update the buffer for the next iteration
        if i + 1 < max_len:
            decoder_input[0, i + 1, next_idx] = 1.0

    # Map to String
    result_str = "".join([index_to_char[idx] for idx in predicted_indices 
                          if index_to_char[idx] not in [start_token, end_token]])
    
    return result_str

In [31]:
# Evaluate 10 random input images
start = random.randint(0,1000)
stop = start+10
sample_imgs = X_test[start: stop]
a = decode_labels_tf(y_test[start:stop])
expressions = []
for img in sample_imgs:
    expression = predict_math_expression(img, image2text, reverse_indices)
    expressions.append(expression)

print(a, expressions)

['80 ', '156', '94 ', '-78', '-18', '116', '43 ', '78 ', '109', '143'] ['70 ', '156', '84 ', '-78', '-18', '116', '45 ', '78 ', '109', '70 ']


#### Batch inference or something

In [40]:
# Batch inference. This is faster.
def evaluate_calculator_batch(test_images, test_targets, vision_model, calc_model, 
                              index_to_char, start_token='<start>', end_token='<end>', 
                              max_len=4, vocab_size=15):

    num_samples = test_images.shape[0]
    
    # Visual encoder
    zero_array = np.zeros((num_samples, 6, vocab_size))
    expression_probs = vision_model.predict([test_images, zero_array], batch_size=32, verbose=1)
    
    # Calculator Phase: Recursive decoding in a batch loop
    char_to_index = {v: k for k, v in index_to_char.items()}
    start_idx = char_to_index[start_token]
    end_idx = char_to_index[end_token]
    
    decoded_answer = np.zeros((num_samples, max_len, vocab_size))
    decoded_answer[:, 0, start_idx] = 1.0
    
    # Track which samples in the batch have hit the end token
    finished = np.zeros(num_samples, dtype=bool)
    final_indices = np.full((num_samples, max_len), end_idx)
    final_indices[:, 0] = start_idx

    for i in range(max_len - 1):
        # We only run the calculator model here (very fast)
        preds = calc_model.predict([expression_probs, decoded_answer], batch_size=num_samples, verbose=0)
        
        # Look at the prediction for the NEXT character
        next_indices = np.argmax(preds[:, i, :], axis=-1)
        
        for b_idx in range(num_samples):
            if not finished[b_idx]:
                token = next_indices[b_idx]
                final_indices[b_idx, i+1] = token
                decoded_answer[b_idx, i+1, token] = 1.0
                if token == end_idx:
                    finished[b_idx] = True
        
        if np.all(finished): break

    # Calculation of Metrics
    true_indices = np.argmax(test_targets, axis=-1)
    
    token_correct = 0
    math_correct = 0
    
    for i in range(num_samples):
        p_str = "".join([index_to_char[idx] for idx in final_indices[i] if index_to_char[idx] not in [start_token, end_token, ' ']])
        t_str = "".join([index_to_char[idx] for idx in true_indices[i] if index_to_char[idx] not in [start_token, end_token, ' ']])
        
        # Math Accuracy
        if p_str == t_str:
            math_correct += 1
            
        # Token Accuracy
        for t_step in range(max_len):
            if final_indices[i, t_step] == true_indices[i, t_step]:
                token_correct += 1
                
    total_tokens = num_samples * max_len
    
    print(f"\nFinal Math Correctness: {math_correct/num_samples:.2%}")
    print(f"Final Token Accuracy: {token_correct/total_tokens:.2%}")
    
    return math_correct/num_samples, token_correct/total_tokens

In [41]:
# Use the fast batch evaluator

X_sample_set = X_test
y_sample_set = y_test

math_acc, token_acc = evaluate_calculator_batch(
    X_sample_set, y_sample_set, 
    image2text_pretraining, text2text_calculator, 
    reverse_indices
)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step

Final Math Correctness: 78.00%
Final Token Accuracy: 90.91%


In [27]:
image2text.save(f'math_acc_{math_acc:3f}.keras')

In [ ]:
import numpy as np

def inference_vision_batch(vision_model, image_sequences, batch_size=32):
    """
    Applies vectorized inference to transform images into expression probabilities.
    image_sequences: Shape (N, 5, 28, 28, 1)
    """
    num_samples = image_sequences.shape[0]
    
    # We provide a dummy input for the teacher-forcing branch (input_2)
    # Since the Vision model's output doesn't rely on this during inference
    dummy_input = np.zeros((num_samples, 6, 15)) 
    
    # Predict in batches to avoid OOM (Out of Memory) errors
    expression_probs = vision_model.predict(
        [image_sequences, dummy_input], 
        batch_size=batch_size, 
        verbose=0
    )
    
    return expression_probs

In [42]:
image2text_pretraining.summary(expand_nested=True, line_length=120)

print("\n--- CALCULATOR SUMMARY ---")
text2text_calculator.summary(expand_nested=True, line_length=120)

Model: "pretraining_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━
┃ Layer (type)                      ┃ Output Shape                 ┃           Param # ┃ Connected to              
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━
│ sequence (InputLayer)             │ (None, 5, 28, 28, 1)         │                 0 │ -                         
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ encoder_model (Functional)        │ [(None, 5, 7, 7, 128),       │           906,560 │ sequence[0][0]            
│                                   │ (None, 7, 7, 128), (None, 7, │                   │                           
│                                   │ 7, 128)]                     │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│    └ input_layer_5 (InputLayer)   │ (None, 5, 28, 28, 1)         │                 0 │ -                         
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│    └ time_distributed_26          │ (None, 5, 28, 28, 32)        │               320 │ -                         
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│    └ time_distributed_27          │ (None, 5, 28, 28, 32)        │               128 │ -                         
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│    └ time_distributed_28          │ (None, 5, 28, 28, 32)        │                 0 │ -                         
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│    └ time_distributed_29          │ (None, 5, 28, 28, 32)        │                 0 │ -                         
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│    └ time_distributed_30          │ (None, 5, 14, 14, 32)        │                 0 │ -                         
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│    └ time_distributed_32          │ (None, 5, 14, 14, 64)        │            18,496 │ -                         
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│    └ time_distributed_33          │ (None, 5, 14, 14, 64)        │               256 │ -                         
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│    └ time_distributed_34          │ (None, 5, 14, 14, 64)        │                 0 │ -                         
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼───────────────────

 Total params: 3,764,591 (14.36 MB)

 Trainable params: 1,254,799 (4.79 MB)

 Non-trainable params: 192 (768.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━
┃ Layer (type)                      ┃ Output Shape                 ┃           Param # ┃ Connected to              
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━
│ expression_input (InputLayer)     │ (None, 6, 15)                │                 0 │ -                         
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ answer (InputLayer)               │ (None, 4, 15)                │                 0 │ -                         
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ calculator_encoder (LSTM)         │ [(None, 6, 256), (None,      │           278,528 │ expression_input[0][0]    
│                                   │ 256), (None, 256)]           │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ decoder_lstm (LSTM)               │ [(None, 4, 256), (None,      │           278,528 │ answer[0][0],             
│                                   │ 256), (None, 256)]           │                   │ calculator_encoder[0][1], 
│                                   │                              │                   │ calculator_encoder[0][2]  
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ attention_block (Attention)       │ (None, 4, 256)               │                 0 │ decoder_lstm[0][0],       
│                                   │                              │                   │ calculator_encoder[0][0]  
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ concat_q_A (Concatenate)          │ (None, 4, 512)               │                 0 │ decoder_lstm[0][0],       
│                                   │                              │                   │ attention_block[0][0]     
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ combined_dense (Dense)            │ (None, 4, 256)               │           131,328 │ concat_q_A[0][0]          
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ decoder_res_dense                 │ (None, 4, 256)               │             3,840 │ answer[0][0]              
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ decoder_with_residual (Add)       │ (None, 4, 256)               │                 0 │ combined_dense[0][0],     
│                                   │                              │                   │ decoder_res_dense[0][0]   
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ decoder_activation                │ (None, 4, 256)               │                 0 │ decoder_with_residual[0][0
│ (TimeDistributed)                 │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ decoder_layer_norm                │ (None, 4, 256)               │               512 │ decoder_activation[0][0]  
│ (LayerNormalization)              │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ decoder_dense (TimeDistributed)   │ (None, 4, 15)     

 Total params: 2,089,775 (7.97 MB)

 Trainable params: 696,591 (2.66 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,393,184 (5.31 MB)